## Demonstrating SAAS Functionality in Xopt GP Models
This example compares two models trained on the same sparse problem:
1. A standard GP model.
2. A GP model with SAAS priors enabled via `saas_outputs`.

The synthetic objective depends only on `x0` and `x1`, while the remaining dimensions are irrelevant. This makes it easy to see how SAAS encourages sparsity in learned lengthscales.

In [ ]:
# set values if testing
import os

import matplotlib.pyplot as plt
import pandas as pd

from xopt import Xopt, Evaluator
from xopt.generators import RandomGenerator
from xopt.resources.test_functions.rosenbrock import make_rosenbrock_vocs
from xopt.generators.bayesian.models import StandardModelConstructor
from xopt.generators.bayesian.visualize import visualize_model

SMOKE_TEST = os.environ.get("SMOKE_TEST")

# Use a higher-dimensional input space to make sparsity visible.
n_dim = 20
n_initial = 15
vocs = make_rosenbrock_vocs(n_dim)
vocs.observables = ["z"]


def evaluate_func(X):
    return {
        "y": X["x0"] ** 2 + X["x1"] ** 2 + X["x10"] ** 2,
        "z": X["x0"] ** 2,
    }

## Generate Training Data

In [ ]:
evaluator = Evaluator(function=evaluate_func)
generator = RandomGenerator(vocs=vocs)
X = Xopt(generator=generator, evaluator=evaluator)
X.random_evaluate(n_initial)

data = X.data
data[vocs.variable_names + ["y", "z"]].head()

## Build Standard and SAAS Models
Both models are trained on the same data so any difference in behavior comes from the SAAS prior configuration.

In [ ]:
standard_constructor = StandardModelConstructor()
saas_constructor = StandardModelConstructor(saas_outputs=["y", "z"])

# Build two models from identical data.
standard_model = standard_constructor.build_model(
    input_names=vocs.variable_names,
    outcome_names=["y", "z"],
    data=data,
)

saas_model = saas_constructor.build_model(
    input_names=vocs.variable_names,
    outcome_names=["y", "z"],
    data=data,
)

In [ ]:
def get_lengthscale_vector(gp_model):
    for name, value in gp_model.named_parameters():
        if "lengthscale" in name:
            return value.detach().cpu().numpy().ravel()
    raise RuntimeError("No lengthscale parameter found")


standard_y_model = standard_model.models[vocs.output_names.index("y")]
saas_y_model = saas_model.models[vocs.output_names.index("y")]

comparison = pd.DataFrame(
    {
        "variable": vocs.variable_names,
        "standard_lengthscale": get_lengthscale_vector(standard_y_model),
        "saas_lengthscale": get_lengthscale_vector(saas_y_model),
    }
)

comparison.sort_values("saas_lengthscale")

## Compare Learned Lengthscales
Smaller lengthscales correspond to stronger local sensitivity for that variable.
Because this objective is sparse, SAAS should concentrate sensitivity on `x0` and `x1` 
while the standard model does not until there is a significant amount of data.

In [ ]:
plot_data = comparison.set_index("variable")[
    ["standard_lengthscale", "saas_lengthscale"]
]

ax = plot_data.plot(kind="bar", figsize=(10, 4), rot=0)
ax.set_ylabel("Lengthscale")
ax.set_title("Standard GP vs SAAS GP lengthscales for output y")
plt.tight_layout()

In [ ]:
reference_point = data.iloc[-1][vocs.variable_names].to_dict()

fig, ax = visualize_model(
    standard_model,
    vocs,
    data,
    variable_names=["x0", "x1"],
    reference_point=reference_point,
)
fig.suptitle("Standard GP", y=1.02)

fig, ax = visualize_model(
    saas_model,
    vocs,
    data,
    variable_names=["x0", "x1"],
    reference_point=reference_point,
)
fig.suptitle("SAAS GP", y=1.02)

## Add more data and rebuild models to see how lengthscales evolve.

In [ ]:
X.random_evaluate(n_initial)

# rebuild models with more data to see how lengthscales evolve
standard_model = standard_constructor.build_model(
    input_names=vocs.variable_names,
    outcome_names=["y", "z"],
    data=X.data,
)

saas_model = saas_constructor.build_model(
    input_names=vocs.variable_names,
    outcome_names=["y", "z"],
    data=X.data,
)

In [ ]:
standard_y_model = standard_model.models[vocs.output_names.index("y")]
saas_y_model = saas_model.models[vocs.output_names.index("y")]

comparison = pd.DataFrame(
    {
        "variable": vocs.variable_names,
        "standard_lengthscale": get_lengthscale_vector(standard_y_model),
        "saas_lengthscale": get_lengthscale_vector(saas_y_model),
    }
)

comparison.sort_values("saas_lengthscale")

plot_data = comparison.set_index("variable")[
    ["standard_lengthscale", "saas_lengthscale"]
]

ax = plot_data.plot(kind="bar", figsize=(10, 4), rot=0)
ax.set_ylabel("Lengthscale")
ax.set_title("Standard GP vs SAAS GP lengthscales for output y")
plt.tight_layout()

## Takeaway
SAAS is enabled by setting `saas_outputs` in `StandardModelConstructor`.
In sparse problems, SAAS generally produces more selective lengthscales and can improve robustness when many dimensions are weakly relevant.